In [1]:
import os
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization, Lambda
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# -----------------------------
# Data Loading and Preprocessing
# -----------------------------
# Define the paths to your dataset directories (we use only real ECG data for identity training)
DATA_DIR_REAL = "C:/Users/M2-Winterfell/Downloads/GAN-models-for-Bio-Authentication-through-ECG-signals/datasets/real_ecgs"

# Function to load and extract Lead I from a single .asc file
def load_lead_I_from_asc(file_path):
    try:
        # Load the .asc file (assuming it's space or tab-delimited)
        ecg_data = np.loadtxt(file_path)
        # Extract Lead I (assuming first column is Lead I)
        lead_I = ecg_data[:, 0]
        return lead_I
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

# Function to load exactly 2000 Lead I ECG data from a specified directory with a progress bar
def load_exactly_2000_lead_I_from_directory(data_dir, n=2000):
    all_lead_I = []
    i = 0
    with tqdm(total=n, desc=f"Loading Lead I from {os.path.basename(data_dir)}", unit='file') as pbar:
        while len(all_lead_I) < n:
            file_name = f"{i}.asc"
            file_path = os.path.join(data_dir, file_name)
            if os.path.exists(file_path):
                lead_I = load_lead_I_from_asc(file_path)
                if lead_I is not None:
                    all_lead_I.append(lead_I)
                    pbar.update(1)
            i += 1
    return np.array(all_lead_I)

# Load 2000 real ECG records (we assume these come from registered users)
lead_I_real = load_exactly_2000_lead_I_from_directory(DATA_DIR_REAL, n=2000)

# Normalize each ECG signal to the range [-1, 1] using min-max normalization
def min_max_normalize(data):
    min_val = np.min(data)
    max_val = np.max(data)
    return 2 * (data - min_val) / (max_val - min_val) - 1

lead_I_real = np.array([min_max_normalize(ecg) for ecg in lead_I_real])

print('Real ECG Shape:      ', lead_I_real.shape)

Loading Lead I from real_ecgs: 100%|██████████| 2000/2000 [00:37<00:00, 53.35file/s]


Real ECG Shape:       (2000, 5000)


In [2]:
# ---------------------------------------------------
# Create Identity Labels for the Registered Users
# ---------------------------------------------------
# For demonstration, we simulate that the 2000 real ECGs come from 2000 different users.
# (e.g., 1 samples per user)
num_registered_users = 2000

# Create user ID labels: 0, 1, 2, …, num_registered_users-1
y_identity = np.arange(num_registered_users)

# Use only the real ECGs and their identity labels for training the feature embedding model.
X = lead_I_real  # Shape: (2000, samples_per_record)
y = y_identity   # Identity labels (0 to num_registered_users-1)

# Split the data into training and test sets (with shuffling)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42, shuffle=True)

# For 1D CNN, reshape each ECG signal to (signal_length, 1)
INPUT_SHAPE = (X_train.shape[1], 1)
X_train = X_train.reshape(-1, INPUT_SHAPE[0], 1)
X_test  = X_test.reshape(-1, INPUT_SHAPE[0], 1)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (1800, 5000, 1)
X_test shape: (200, 5000, 1)


In [3]:
# ---------------------------------------------------
# Build the CNN Feature Embedding Model with a Temporary Classification Head
# ---------------------------------------------------
embedding_dim = 128  # dimension of the embedding

base_model = Sequential()
# Block 1
base_model.add(Conv1D(filters=32, kernel_size=3, activation='relu', padding='same', input_shape=INPUT_SHAPE))
base_model.add(BatchNormalization())
base_model.add(Conv1D(filters=32, kernel_size=3, activation='relu', padding='same'))
base_model.add(BatchNormalization())
base_model.add(MaxPooling1D(pool_size=2))
base_model.add(Dropout(0.25))
# Block 2
base_model.add(Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'))
base_model.add(BatchNormalization())
base_model.add(Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'))
base_model.add(BatchNormalization())
base_model.add(MaxPooling1D(pool_size=2))
base_model.add(Dropout(0.25))
# Block 3
base_model.add(Conv1D(filters=128, kernel_size=3, activation='relu', padding='same'))
base_model.add(BatchNormalization())
base_model.add(Conv1D(filters=128, kernel_size=3, activation='relu', padding='same'))
base_model.add(BatchNormalization())
base_model.add(MaxPooling1D(pool_size=2))
base_model.add(Dropout(0.25))

base_model.add(Flatten())
base_model.add(Dense(128, activation='relu'))
base_model.add(Dropout(0.25))
# Embedding layer: outputs a 128-dim vector.
base_model.add(Dense(embedding_dim, activation=None))
# L2 normalize the embeddings.
base_model.add(Lambda(lambda x: tf.math.l2_normalize(x, axis=1)))
# Temporary classification head for training
base_model.add(Dense(num_registered_users, activation='softmax'))

optimizer = Adam(learning_rate=0.005)
base_model.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

# Train the model
batch_size = 32
history = base_model.fit(X_train, y_train, 
                         epochs=50, 
                         batch_size=batch_size,
                         validation_data=(X_test, y_test))

test_loss, test_acc = base_model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc*100:.2f}%")

# Save the trained model (including the temporary head)
base_model.save("my_ecg_identity_model.h5")
print("Model saved successfully.")

Epoch 1/50
57/57 [==============================] - 49s 784ms/step - loss: 7.6461 - accuracy: 0.0000e+00 - val_loss: 7.6701 - val_accuracy: 0.0000e+00
Epoch 2/50
57/57 [==============================] - 44s 769ms/step - loss: 7.7266 - accuracy: 0.0000e+00 - val_loss: 7.9389 - val_accuracy: 0.0000e+00
Epoch 3/50
57/57 [==============================] - 44s 774ms/step - loss: 7.5462 - accuracy: 0.0000e+00 - val_loss: 9.0834 - val_accuracy: 0.0000e+00
Epoch 4/50
57/57 [==============================] - 43s 762ms/step - loss: 7.3849 - accuracy: 0.0000e+00 - val_loss: 9.1907 - val_accuracy: 0.0000e+00
Epoch 5/50
57/57 [==============================] - 42s 745ms/step - loss: 7.2507 - accuracy: 0.0011 - val_loss: 9.3886 - val_accuracy: 0.0000e+00
Epoch 6/50
57/57 [==============================] - 42s 740ms/step - loss: 7.1151 - accuracy: 0.0028 - val_loss: 9.6293 - val_accuracy: 0.0000e+00
Epoch 7/50
57/57 [==============================] - 43s 747ms/step - loss: 7.0003 - accuracy: 0.0011 -

c:\Users\M2-Winterfell\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\engine\training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model, Model

# Load the saved model (includes the temporary classification head)
loaded_model = load_model("C:/Users/M2-Winterfell/Downloads/my_ecg_identity_model.h5")

# Create an embedding model that outputs the 128-dim normalized feature vector.
# Here, we assume that the second-to-last layer (index -2) is the L2 normalization layer.
embedding_model = Model(inputs=loaded_model.input, outputs=loaded_model.layers[-2].output)

# -----------------------------
# Registration Phase
# -----------------------------
# Since each real ECG corresponds to one unique user, compute and store its embedding.
# For demonstration, we use the training set as our registered database.
registered_embeddings = embedding_model.predict(X_train)
# The index of each embedding corresponds to a registered user ID (0 to 1999)

# -----------------------------
# Verification Phase
# -----------------------------
def verify_ecg_signal(ecg_signal, registered_embeddings, threshold=0.8):
    """
    ecg_signal: a single ECG sample with shape (1, signal_length, 1)
    registered_embeddings: numpy array of shape (num_registered_users, embedding_dim)
    threshold: cosine similarity threshold for authentication
    Returns: (authenticated (bool), best_similarity (float))
    """
    new_embedding = embedding_model.predict(ecg_signal)  # shape: (1, embedding_dim)
    new_embedding = new_embedding[0]
    # Compute cosine similarities between new_embedding and each registered embedding.
    # (Since embeddings are L2-normalized, the dot product equals cosine similarity.)
    similarities = np.dot(registered_embeddings, new_embedding)
    best_similarity = np.max(similarities)
    authenticated = best_similarity > threshold
    return authenticated, best_similarity

57/57 [==============================] - 6s 109ms/step


In [18]:
# -----------------------------
# Example Verification
# -----------------------------
# Load a new ECG signal (from a .asc file) to verify its authenticity.
ecg_file_path = "C:/Users/M2-Winterfell/Downloads/pulse2pulse_150k/from_006_chkp_2500_150k/2210.asc"
              # "C:/Users/M2-Winterfell/Downloads/pulse2pulse_fake/fake_ecg_998.asc"
              # "C:/Users/M2-Winterfell/Downloads/GAN-models-for-Bio-Authentication-through-ECG-signals/datasets/real_ecgs/0.asc"
              # "C:/Users/M2-Winterfell/Downloads/GAN-models-for-Bio-Authentication-through-ECG-signals/datasets/cnn_fake/0.asc"
              # "C:/Users/M2-Winterfell/Downloads/pulse2pulse_150k/from_006_chkp_2500_150k/220.asc"
ecg_data = np.loadtxt(ecg_file_path)

if ecg_data.ndim > 1:
    ecg_data = ecg_data[:, 0]

ecg_data = min_max_normalize(ecg_data)

# Reshape to (1, signal_length, 1)
ecg_data = ecg_data.reshape(1, 5000, 1)

# Verify the ECG signal
authenticated, best_similarity = verify_ecg_signal(ecg_data, registered_embeddings, threshold=0.8)
prediction_label = "Real" if authenticated else "Fake"
print(f"Prediction: {prediction_label} (Best Cosine Similarity: {best_similarity:.2f})")


1/1 [==============================] - 0s 25ms/step
Prediction: Fake (Best Cosine Similarity: 0.59)
